In [ ]:
import os

for dirname, dirnames, filenames in os.walk('/kaggle/input/'):
    # Print the current folder
    print(f'Folder: {dirname}')
    
    # Print all files in this folder with an indentation
    for filename in filenames:
        print(f'    File: {filename}')

In [ ]:
import cv2
# /kaggle/input/competitions/image-to-image
img = cv2.imread('/kaggle/input/competitions/image-to-image/train/173.png', cv2.IMREAD_GRAYSCALE)
height, width = img.shape
print(f"Height (Rows): {height}, Width (Columns): {width}")

In [ ]:
!pip install segmentation-models-pytorch

In [ ]:
!pip install pytorch-msssim

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms.functional as TF
import random

TRAIN_PATH = '/kaggle/input/competitions/image-to-image/train/'
CLEAN_PATH = '/kaggle/input/competitions/image-to-image/train_cleaned/'
TEST_PATH = '/kaggle/input/competitions/image-to-image/test/'
IMG_SIZE = (256, 256)
BATCH_SIZE = 16
EPOCHS = 60
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def calculate_score(mse_loss):
    if mse_loss <= 1e-10:
        return 100.0
    return -10 * np.log10(mse_loss)

def get_submission_filename():
    k = 1
    while os.path.exists(f"submission_{k}.csv"):
        k += 1
    return f"submission_{k}.csv"

class DenoiseDataset(Dataset):
    def __init__(self, img_path, clean_path=None, augment=False):
        self.img_path = img_path
        self.clean_path = clean_path
        self.use_augment = augment
        self.filenames = sorted([f for f in os.listdir(img_path) if f.endswith('.png')])

    def augment(self, img):
        if random.random() < 0.5:
            k = random.choice([3, 5])
            img = TF.gaussian_blur(img, kernel_size=k)

        if random.random() < 0.5:
            brightness = 1.0 + random.uniform(-0.15, 0.15)
            contrast = 1.0 + random.uniform(-0.15, 0.15)
            img = TF.adjust_brightness(img, brightness)
            img = TF.adjust_contrast(img, contrast)

        if random.random() < 0.3:
            gamma = random.uniform(0.9, 1.1)
            img = TF.adjust_gamma(img, gamma)

        if random.random() < 0.25:
            noise = torch.randn_like(img) * 0.02
            img = torch.clamp(img + noise, 0.0, 1.0)

        return img

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        f = self.filenames[idx]

        img = cv2.imread(os.path.join(self.img_path, f), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, IMG_SIZE).astype(np.float32) / 255.0
        img_tensor = torch.from_numpy(img).unsqueeze(0)

        if self.use_augment:
            img_tensor = self.augment(img_tensor)

        if self.clean_path:
            target = cv2.imread(os.path.join(self.clean_path, f), cv2.IMREAD_GRAYSCALE)
            target = cv2.resize(target, IMG_SIZE).astype(np.float32) / 255.0
            target_tensor = torch.from_numpy(target).unsqueeze(0)
            return img_tensor, target_tensor, f

        return img_tensor, f

class ResidualProUNet(nn.Module):
    def __init__(self):
        super(ResidualProUNet, self).__init__()

        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True)
            )

        self.enc1 = block(1, 32)
        self.enc2 = block(32, 64)
        self.enc3 = block(64, 128)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = block(128, 256)
        self.drop = nn.Dropout2d(0.2)

        self.up1 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec1 = block(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = block(128, 64)
        self.up3 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec3 = block(64, 32)

        self.final = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x)
        p1 = self.pool(s1)
        s2 = self.enc2(p1)
        p2 = self.pool(s2)
        s3 = self.enc3(p2)
        p3 = self.pool(s3)

        b = self.bottleneck(p3)
        b = self.drop(b)

        u1 = self.up1(b)
        u1 = torch.cat([u1, s3], dim=1)
        d1 = self.dec1(u1)

        u2 = self.up2(d1)
        u2 = torch.cat([u2, s2], dim=1)
        d2 = self.dec2(u2)

        u3 = self.up3(d2)
        u3 = torch.cat([u3, s1], dim=1)
        d3 = self.dec3(u3)

        noise_map = self.final(d3)
        return torch.clamp(x - noise_map, 0, 1)

full_dataset = DenoiseDataset(TRAIN_PATH, CLEAN_PATH, augment=True)
train_size = int(0.85 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

model = ResidualProUNet().to(DEVICE)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=5)

for epoch in range(EPOCHS):
    model.train()
    t_mse = 0

    for imgs, targets, _ in train_loader:
        imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), targets)
        loss.backward()
        optimizer.step()
        t_mse += loss.item()

    avg_t_mse = t_mse / len(train_loader)

    model.eval()
    v_mse = 0

    with torch.no_grad():
        for v_imgs, v_targets, _ in val_loader:
            v_out = model(v_imgs.to(DEVICE))
            v_mse += criterion(v_out, v_targets.to(DEVICE)).item()

    avg_v_mse = v_mse / len(val_loader)
    scheduler.step(avg_v_mse)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_t_mse:.6f} | Val Score: {calculate_score(avg_v_mse):.2f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

print("Creating submission file...")
test_ds = DenoiseDataset(TEST_PATH, augment=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)
model.eval()

submission_data = []

with torch.no_grad():
    for img, filename in test_loader:
        img_id = filename[0].replace('.png', '')
        orig = cv2.imread(os.path.join(TEST_PATH, filename[0]), cv2.IMREAD_GRAYSCALE)
        h, w = orig.shape
        pred = model(img.to(DEVICE)).cpu().squeeze().numpy()
        pred_full = cv2.resize(pred, (w, h))

        for r in range(h):
            for c in range(w):
                submission_data.append([f"{img_id}_{r+1}_{c+1}", pred_full[r, c]])

output_file = get_submission_filename()
pd.DataFrame(submission_data, columns=['id', 'value']).to_csv(output_file, index=False)
print(f"Saved as {output_file}")